In [4]:
import os
from llama_parse import LlamaParse
from langchain_core.documents import Document as LC_Document
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np

Load the Data and Parsing the data

In [ ]:
api_lp_key = ""

parser = LlamaParse(
    api_key = api_lp_key,
    result_type = "markdown", # For simple documents you can text instead of markdown. Markdown is more reliable when reading the tables and images
    num_workers = 2,
    verbose = True,
    language = "en"
)

file_path = "./file_survey_paper.pdf"
documents = parser.load_data(file_path)


Started parsing the file under job_id 05c885ff-ade4-4660-9f79-103ae7c8c836


In [6]:
print(f"Loaded {len(documents)} document pages/sections.")
print(f"Sample Content:\n{documents[0].text[:500]}...")

Loaded 144 document pages/sections.
Sample Content:
arXiv:2303.18223v19 [cs.CL] 18 Mar 2026

# A Survey of Large Language Models

Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen Zhang, Junjie Zhang, Zican Dong, Yifan Du, Chen Yang, Yushuo Chen, Zhipeng Chen, Jinhao Jiang, Ruiyang Ren, Yifan Li, Xinyu Tang, Zikang Liu, Peiyu Liu, Jian-Yun Nie and Ji-Rong Wen†

# Abstract

Ever since the Turing Test was proposed in the 1950s, humans have explored the mastering of language intelligence by machine. L...


Using recursive splitting and Semantic chunking

In [7]:
embed_model = OllamaEmbeddings(model="nomic-embed-text")


# convert LlamaIndex Documents -> LangChain Documents
lc_documents = [
    LCDocument(
        page_content=doc.text,
        metadata=doc.metadata
    )
    for doc in documents
]

# Before doing the semantic chunking. I am doing additional chunking. so, a chunk can't have more than 8192 tokens. which is maximum for model.
pre_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,       
    chunk_overlap=200,     # small overlap to preserve context at boundaries
)
pre_split_docs = pre_splitter.split_documents(lc_documents)
print(f"Pre-split into {len(pre_split_docs)} chunks before semantic chunking.")


chunks = SemanticChunker(
    embed_model,
    breakpoint_threshold_type="percentile", 
    breakpoint_threshold_amount=95.0         # tune: lower = more chunks, higher = fewer
)


semantic_chunks = chunks.split_documents(pre_split_docs)

NameError: name 'LCDocument' is not defined

In [ ]:
print(f"Created {len(semantic_chunks)} semantic chunks using nomic-embed-text.")
print(f"Example Chunk Content:\n{semantic_chunks[0].page_content[:200]}...")